# Scrape Character Data from *One Piece: Voyage ([航海王启航](https://op.mobage.cn/wiki))*

Jan 14, 2021

## Ratings img
https://cn-opweb-cdn.mobage.cn/cdn/wiki/op/static/img/attr.png
![ratings: b, a, s, ss, ss+, sss, sss+](https://cn-opweb-cdn.mobage.cn/cdn/wiki/op/static/img/attr.png)


## img link explanation:
skin img (li_hui): `"https://cn-opweb-cdn.mobage.cn/cdn/wiki/op/static/img/li_hui/{skin}.png"`

skill img: `"https://cn-opweb-cdn.mobage.cn/cdn/wiki/op/static/img/skill/{skill}.png"`

In [2]:
# luffy             : 10001
# new world huojinsi: 10480
# date: Jan 14 2021

import OPVoyageScraper

start = 10001
end   = 10500
OPVoyageScraper.scrape(start, end)

[====================] 100%	(500/500)

In [2]:
import bs4
import sys
import requests
import re
import pandas as pd

In [3]:
chaId = 10418 # new world Zoro
chaId = 10275 # <empty>
# chaId = 10433 # justice flower tashigi: no skin
# chaId = 10075 # boa hancock: 2 skins
# chaId = 10417 # big mom: 6 skills
# chaId = 10161 # law: zsbz
# chaId = 10006 # shanks: no article
chaId = 10413 # 

# anti-scraping
user_agent = "Mozilla/5.0 (Windows NT 10.0; WOW64; rv:68.0) Gecko/20100101 Firefox/68.0"
url = f'https://op.mobage.cn/wiki/role/{chaId}'
response = requests.get(url, headers={'User-Agent': user_agent})
if response.status_code == 200:
    # print(response.content.decode('utf-8'))
    content = response.content.decode('utf-8')
else:
    print(f"Fail to get the url [{chaId}, {response.status_code}]")

In [4]:
# start reading content
soup = bs4.BeautifulSoup(content, 'html.parser')

# save all data in a json node
node = {}

In [5]:
## role-bg
name = soup.find('img', id='li_hui')['alt']
img = soup.find('img', id='li_hui')['src'].split('img/')[1]
rating = soup.find('div', class_='img-attr').attrs['class'][1]
tags = soup.find('div', class_='role-tags').get_text(' ').split()
show = soup.find('p', class_='role-show').get_text()
roleEquip = soup.find('p', class_='role-equip').get_text(' ').split()[1]
way = soup.find('p', class_='role-way').get_text(' ').split()[1]

node = {
    '人物': name,
    '立绘': img,
    '头像': None,
    '级别': rating,
    '标签': tags,
    '图鉴': show,
    '专属': {
        roleEquip: None,},
    '获取途径': way}

# optional parts: tupo, skin
tupo = soup.find('li', class_='tupo-bg')
if tupo.find('a'):
    tupoId = int(tupo.find('a')['href'].split('/')[-1])
else:
    # no tupo
    tupoId = None
node['突破人物'] = tupoId
    
skins = []
for skin in soup.findAll('li', class_='skin-bg'):
    skins.append(skin['data-img'].split('img/')[1])
node['皮肤'] = skins

## part-1
attrs = soup.find('ul', class_='clear').get_text(' ').split()
attrNames = [s.replace('：','').strip() for s in attrs[0::2]]
attrValues = [float(s) for s in attrs[1::2]]
attrDict = {}
for i in range(len(attrNames)):
    attrDict[attrNames[i]] = attrValues[i]
node['属性'] = attrDict
    
# skill
base = soup.find('div', class_='base-skill')
skillImgs = [u.attrs['style'].split('img/')[1][:-1]
             for u in base.findAll('span')[1:]]
skillList = [p.get_text() for p in base.findAll('p')]
skillNames = skillList[0::2]
skillDesc = skillList[1::2]
skillDict = {}
for i in range(len(skillNames)):
    skillDict[skillNames[i]] = {"desc": skillDesc[i], "img": skillImgs[i]}
node['技能'] = skillDict
    
team = soup.find('div', class_='base-team')
teamCount = len(team.findAll('span', class_='title-span'))
teamP = team.findAll('p')
if teamCount == 2:
    # team, gem
    teamList = [p.get_text() for p in teamP[:5]]
    teamP = teamP[5:]
    node['阵容方案'] = teamList
elif teamCount != 1:
    raise ValueError("Unknown Team Description")
    
# gem
gems = [p.get_text() for p in teamP]
node['共鸣方案'] = gems

# equip
equipDict = {}
for equip in soup.findAll('div', class_='equip-part'):
    equipText = equip.get_text(' ').split()
    add = {}
    for e in equipText[1:]:
        string = e.split('+')
        add[string[0]] = int(string[1])
    equipDict[equipText[0]] = add
node['装备'] = equipDict

roleEquipImg = soup.findAll('span', class_='equip')[4].attrs['style'].split('img/')[1][:-1]
node['专属'][roleEquip] = roleEquipImg

jxxg = soup.find('div', class_='taps-jxxg')
jxxgDict = {}
jxLevel = jxxg.find('div', class_='child-tap').get_text(" ").split()
jxAddText = jxxg.findAll('div', class_='add-value')
jxEffectText = jxxg.findAll('div', class_='flex-part')
jxEffect = [flex.find('p').get_text() for flex in jxEffectText][1::3]
for level in range(len(jxLevel)):
    jxAdd = {}
    for i in jxAddText[level].get_text(' ').split()[1:]:
        string = i.split('：')
        jxAdd[string[0]] = int(string[1])
    jxxgDict[jxLevel[level]] = {"成长值": jxAdd, "效果": jxEffect[level]}
node['觉醒'] = jxxgDict

avatar = jxxg.findAll('div', class_='avatar-bg')[1].find('img')['src'].split('img/')[1]
node['头像'] = avatar

doctor = soup.find('div', class_='taps-chuanyi').find('p').get_text().split('+')
node['船医'] = {doctor[0]: doctor[1]}

# optional: chufang, zszb, jiban, tupo
kitchen = soup.find('div', class_='taps-chufang')
if kitchen:
    kitchen = kitchen.find('div', class_='add-value').get_text(' ').split()[1::2]
    kitchenDict = {}
    for k in kitchen:
        ks = k.split(':')
        kitchenDict[ks[0]] = ks[1]
    node['厨房'] = kitchenDict
    

zszb = soup.find('div', class_='taps-zszb')
if zszb:
    zszb = zszb.get_text(' ').split()
    node[zszb[-2].split('：')[0]] = zszb[-1]
    
jiban = soup.find('div', class_='taps-jiban')
if jiban:
    jibanDict = {}
    jibanCount = jiban.find('ul').get_text(' ').split()
    jibans = jiban.findAll('div', class_='part')
    for i in range(len(jibanCount)):
        jDict = {}
        j = jibans[i].get_text(' ').split()
        jDict[j[1]] = j[2]
        people = [txt.split("觉醒")[0] for txt in j[4:-2]]
        jDict["升级条件"] = people
        jibanDict[jibanCount[i]] = jDict
    node['羁绊'] = jibanDict
    
tupo = soup.find('div', class_='taps-tupo')
if tupo:
    tupoDict = {}
    task = [t.get_text(' ').split()[0] for t in tupo.findAll('div', class_='desc-part')]
    tupoDict["突破任务"] = task
    tupoAdd = tupo.findAll('div', class_='w170')[1].get_text(' ').split()[1:]
    tupoAddDict = {}
    for li in [re.split(r'(\d+)', l) for l in tupoAdd]:
        tupoAddDict[li[0]] = int(li[1])
    tupoDict["突破效果LV.60"] = tupoAddDict
    tupoSkill = [s.split('：')[1] 
             for s in tupo.find('div', class_='right-top').get_text(' ').split()]
    tupoDict[tupoSkill[0]] = tupoSkill[1]
    node['突破'] = tupoDict
    
## optional: gonglue
gonglue = soup.find('h2', class_='gonglue')
if gonglue:
    glDict = {}
    gl = gonglue.nextSibling
    titles = gl.findAll('span', class_='red')
    urls = gl.findAll('a')
    for i in range(len(titles)):
        glDict[titles[i].get_text()] = urls[i]['href']
    node['攻略'] = glDict

IndexError: list index out of range

In [6]:
node


{'人物': '七武海·巴奇',
 '立绘': 'li_hui/buggy_03.png',
 '头像': 'hard/10413.png',
 '级别': 'ss',
 '标签': ['敏捷', '后排', '拥有恶魔果实', '物理攻击', '强力输出', '火系', '斩击'],
 '图鉴': '拥有“小丑”外号巴奇海盗团的船长。小时候和杰克斯一起在罗杰海盗团里做见习船员。现在是王下七武海的新成员更是[海盗派遣组织]的总帅。“四分五裂之果”的能力者身体可以分裂。',
 '专属': {'小丑飞刀': 'equip/32413.png'},
 '获取途径': '活动/副本',
 '突破人物': 10474,
 '皮肤': [],
 '属性': {'力量成长': 13.16,
  '物理穿透': 0.0,
  '敏捷成长': 15.82,
  '法术穿透': 0.0,
  '智力成长': 4.41,
  '物理暴击': 1105.68,
  '力量': 1855.4,
  '法术暴击': 62.34,
  '敏捷': 2236.8,
  '物理命中': 90.0,
  '智力': 623.4,
  '法术命中': 90.0,
  '生命值': 145572.2,
  '闪避几率': 328.0,
  '物理攻击': 10259.34,
  '法术抵抗': 328.0,
  '法术攻击': 504.72,
  '生命恢复': 21886.0,
  '物理护甲': 11002.0,
  '能量恢复': 481.0,
  '法术抗性': 4719.8,
  '能量值': 0.0,
  '物理抗暴': 175.0,
  '法术抗暴': 175.0},
 '技能': {'特制马奇弹': {'desc': '对所有敌人造成一次火系物理伤害并击飞，附加易燃效果。3觉时觉醒技会造成2次火系物理伤害，5觉时觉醒技会造成3次火系物理伤害。',
   'img': 'skill/41301.png'},
  '四分五裂·煎饼': {'desc': '七武海·巴奇开场后召唤一名海盗，海盗会释放火箭（对面前所有的敌方造成1次火系物理伤害），当海盗在场时还会使周边的友方的物理护甲和法术抗性提升，并转移七武海·巴奇受到的30%的伤害。海盗本身无法被选中，并霸体。当海盗死亡后，间隔5秒会重新召唤。

In [13]:
tupo = soup.find('div', class_='taps-tupo')
if tupo:
    tupoDict = {}
    taskList = tupo.findAll('div', class_='desc-part')
    if taskList[0].get_text():
        task = [t.get_text(' ').split()[0] for t in taskList]
        tupoDict["突破任务"] = task
    tupoAdd = tupo.findAll('div', class_='w170')[1].get_text(' ').split()[1:]
    tupoAddDict = {}
    for li in [re.split(r'(\d+)', l) for l in tupoAdd]:
        tupoAddDict[li[0]] = int(li[1])
    tupoDict["突破效果LV.60"] = tupoAddDict
    tupoSkill = [s.split('：')[1] 
             for s in tupo.find('div', class_='right-top').get_text(' ').split()]
    tupoDict[tupoSkill[0]] = tupoSkill[1]
    node['突破'] = tupoDict

In [14]:
tupoDict

{'突破效果LV.60': {}, '#N/A': '#N/A'}

In [11]:
tupo.findAll('div', class_='desc-part')

[<div class="desc-part"><div class="avatar-bg f-img"><img alt="" src="https://cn-opweb-cdn.mobage.cn/cdn/wiki/op/static/img/.png"/></div><div class="flex"><p class="add"></p><p></p></div></div>,
 <div class="desc-part"><div class="avatar-bg f-img"><img alt="" src="https://cn-opweb-cdn.mobage.cn/cdn/wiki/op/static/img/.png"/></div><div class="flex"><p class="add"></p><p></p></div></div>]